# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinkk1808/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 — Staleness: OPPOSITE. The youngest pages (0–90 days) had the highest observed decline rate at 66.87%, while pages older than 365 days had the lowest decline rate at 42.63%. Therefore, I will not assume that older content is automatically a stronger refresh candidate.

Signal 2 — CTR vs position: MIXED. Mean CTR generally becomes lower as average position gets worse, but the Top-5 bucket has an unusual median CTR of 0.00. This suggests that CTR and position contain useful information, but a single threshold may not capture the whole pattern.

Baseline rule: Prioritize pages that already receive meaningful impressions, rank within the top 20, but have unusually low CTR for their position. These pages have search visibility but may be underperforming in clicks, so they are useful candidates for human review.

Reason code: LOW_CTR_FOR_POSITION
Action: REVIEW_FOR_CTR_OR_CONTENT_REFRESH

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/pinkk1808/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Outcome used only for checking whether a signal looks useful
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

# -------------------------
# SIGNAL 1: STALENESS
# -------------------------
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[0, 90, 180, 365, np.inf],
    labels=["0-90", "91-180", "181-365", "365+"],
    include_lowest=True
)

age_check = (
    df.groupby("age_bucket", observed=False)
      .agg(
          n=("is_declining", "size"),
          decline_rate=("is_declining", "mean")
      )
      .reset_index()
)

age_check["decline_rate"] = (age_check["decline_rate"] * 100).round(2)

print("SIGNAL 1 — STALENESS")
display(age_check)


# -------------------------
# SIGNAL 2: CTR VS POSITION
# -------------------------
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 5, 10, 20, np.inf],
    labels=["Top 5", "6-10", "11-20", "20+"],
    include_lowest=True
)

ctr_position_check = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          median_ctr=("ctr", "median"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

ctr_position_check["median_ctr"] = ctr_position_check["median_ctr"].round(4)
ctr_position_check["mean_ctr"] = ctr_position_check["mean_ctr"].round(4)

print("SIGNAL 2 — CTR VS POSITION")
display(ctr_position_check)

SIGNAL 1 — STALENESS


,age_bucket,n,decline_rate
0,0-90,492,66.87
1,91-180,11780,62.56
2,181-365,11368,51.49
3,365+,6360,42.63


SIGNAL 2 — CTR VS POSITION


,position_bucket,n,median_ctr,mean_ctr
0,Top 5,5128,0.00,1.2732
1,6-10,9060,0.14,0.5117
2,11-20,7273,0.10,0.3234
3,20+,8539,0.00,0.2113


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import json

# Work only with current observable fields
queue = df.copy()

# Create a safe local identifier for each content-page row
queue["row_id"] = queue.index

# Require some visibility so CTR is meaningful
queue = queue[queue["impressions_90d"] >= 100].copy()

# Expected CTR by broad position tier
def expected_ctr(row):
    if row["avg_position"] <= 5:
        return 0.20
    elif row["avg_position"] <= 10:
        return 0.14
    elif row["avg_position"] <= 20:
        return 0.10
    else:
        return 0.03

queue["expected_ctr"] = queue.apply(expected_ctr, axis=1)

# Difference between expected and observed CTR
queue["ctr_gap"] = (
    queue["expected_ctr"] - queue["ctr"]
).clip(lower=0)

# Give more weight to pages with more impressions
queue["visibility_weight"] = np.log1p(queue["impressions_90d"])

# Final baseline priority score
queue["baseline_score"] = (
    queue["ctr_gap"] * queue["visibility_weight"]
)

# One reason code
queue["reason_code"] = np.where(
    queue["baseline_score"] > 0,
    "LOW_CTR_FOR_POSITION",
    "NO_STRONG_SIGNAL"
)

# Action label
queue["action"] = np.where(
    queue["baseline_score"] > 0,
    "REVIEW_FOR_CTR_OR_CONTENT_REFRESH",
    "MONITOR"
)

# Highest priority first
queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

# Columns for the ranked queue
output_cols = [
    "row_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "expected_ctr",
    "ctr_gap",
    "baseline_score",
    "reason_code",
    "action"
]

ranked_queue = queue[output_cols].copy()

# Write required CSV
os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

# Save a small metrics receipt
metrics = {
    "rows_ranked": int(len(ranked_queue)),
    "top_score": float(ranked_queue["baseline_score"].max()),
    "rule": "LOW_CTR_FOR_POSITION"
}

with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Rows ranked:", len(ranked_queue))
print("CSV written: work/outputs/baseline_action_score.csv")
print("Metrics written: work/outputs/baseline_metrics.json")

display(ranked_queue.head(20))

Rows ranked: 22006
CSV written: work/outputs/baseline_action_score.csv
Metrics written: work/outputs/baseline_metrics.json


,row_id,impressions_90d,ctr,avg_position,expected_ctr,ctr_gap,baseline_score,reason_code,action
0,3331,128068,0.01,2.2,0.20,0.19,2.234462,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
1,7678,272144,0.03,2.3,0.20,0.17,2.127395,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
2,4589,46879,0.01,3.7,0.20,0.19,2.043516,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
3,17362,13676,0.00,4.3,0.20,0.20,1.904694,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
4,28079,33286,0.02,4.6,0.20,0.18,1.874326,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
5,8399,46866,0.03,4.6,0.20,0.17,1.828362,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
6,2559,13341,0.01,3.9,0.20,0.19,1.804748,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
7,17062,11905,0.01,4.8,0.20,0.19,1.783112,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
8,29775,66359,0.04,4.9,0.20,0.16,1.776456,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
9,10459,17713,0.02,3.7,0.20,0.18,1.760780,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_review = ranked_queue.head(20).copy()

def confidence_label(score):
    if score >= 1.90:
        return "HIGH"
    elif score >= 1.75:
        return "MEDIUM-HIGH"
    else:
        return "MEDIUM"

top20_review["confidence_note"] = top20_review["baseline_score"].apply(
    confidence_label
)

top20_review["what_would_make_it_wrong"] = (
    "Low CTR may be normal for the search intent, impressions may be noisy, "
    "or the page/snippet may already be appropriate despite the low CTR."
)

top20_review["review_line"] = top20_review.apply(
    lambda r:
    f"Row {int(r['row_id'])}: "
    f"Action={r['action']} | "
    f"Reason={r['reason_code']} | "
    f"Confidence={r['confidence_note']} "
    f"(score={r['baseline_score']:.3f}, CTR={r['ctr']:.2f}, "
    f"position={r['avg_position']:.1f}, impressions={int(r['impressions_90d'])}) | "
    f"Could be wrong if {r['what_would_make_it_wrong']}",
    axis=1
)

print("TOP-20 MANUAL REVIEW NOTES\n")

for line in top20_review["review_line"]:
    print(line)

TOP-20 MANUAL REVIEW NOTES

Row 3331: Action=REVIEW_FOR_CTR_OR_CONTENT_REFRESH | Reason=LOW_CTR_FOR_POSITION | Confidence=HIGH (score=2.234, CTR=0.01, position=2.2, impressions=128068) | Could be wrong if Low CTR may be normal for the search intent, impressions may be noisy, or the page/snippet may already be appropriate despite the low CTR.
Row 7678: Action=REVIEW_FOR_CTR_OR_CONTENT_REFRESH | Reason=LOW_CTR_FOR_POSITION | Confidence=HIGH (score=2.127, CTR=0.03, position=2.3, impressions=272144) | Could be wrong if Low CTR may be normal for the search intent, impressions may be noisy, or the page/snippet may already be appropriate despite the low CTR.
Row 4589: Action=REVIEW_FOR_CTR_OR_CONTENT_REFRESH | Reason=LOW_CTR_FOR_POSITION | Confidence=HIGH (score=2.044, CTR=0.01, position=3.7, impressions=46879) | Could be wrong if Low CTR may be normal for the search intent, impressions may be noisy, or the page/snippet may already be appropriate despite the low CTR.
Row 17362: Action=REVIEW_

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: The lowest-scoring recommendations in my top 20 are weaker than the first few because their CTR gap and/or impression opportunity is smaller. For example, rows 88, 2803, and 29080 have lower baseline scores than the highest-ranked rows, so I would review them with lower confidence. A recommendation could also be wrong if the low CTR is normal for the search intent, if the impressions are noisy, or if the page already satisfies the user even with a low click rate.

Leakage check: The baseline score uses only impressions_90d, ctr, and avg_position, plus values derived from them. It does not use trend_direction, is_declining, any future-window outcome, or any product flag as an input to the score. I used trend_direction only earlier to audit whether the signals looked reasonable, not to rank the queue. Therefore the ranked baseline is based only on information available at the decision moment.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
score_inputs = {
    "impressions_90d",
    "ctr",
    "avg_position",
    "expected_ctr",
    "ctr_gap",
    "visibility_weight"
}

forbidden_inputs = {
    "trend_direction",
    "is_declining",
    "declined_next_month",
    "future_outcome",
    "product_flag"
}

print("Score inputs:", sorted(score_inputs))
print(
    "Leakage detected:",
    len(score_inputs.intersection(forbidden_inputs)) > 0
)

print("\nWeakest 3 within the top 20:")
display(
    ranked_queue.head(20).tail(3)[
        [
            "row_id",
            "impressions_90d",
            "ctr",
            "avg_position",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
)

Score inputs: ['avg_position', 'ctr', 'ctr_gap', 'expected_ctr', 'impressions_90d', 'visibility_weight']
Leakage detected: False

Weakest 3 within the top 20:


,row_id,impressions_90d,ctr,avg_position,baseline_score,reason_code,action
17,88,12053,0.02,2.6,1.691487,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
18,2803,20324,0.03,3.4,1.686333,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH
19,29080,4446,0.00,2.4,1.679997,LOW_CTR_FOR_POSITION,REVIEW_FOR_CTR_OR_CONTENT_REFRESH


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.